BÀI 2 – TRANG 37–38: KHAI PHÁ LUẬT KẾT HỢP TRÊN DỮ LIỆU ONLINE RETAIL

Câu 1. Đọc dữ liệu vào chương trình



In [ ]:

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules

print("Phạm Lê Anh Khoa - 6451071035")

with open('data/Online Retail.xlsx', 'rb') as f:
    df = pd.read_excel(f)

df.head(12)



Phạm Lê Anh Khoa - 6451071035


In [ ]:
print("Phạm Lê Anh Khoa - 6451071035")
df.info()

In [ ]:
print("Phạm Lê Anh Khoa - 6451071035")
df.describe(include='all')

Câu 2. Tiền xử lý dữ liệu: loại bỏ ký tự thừa ở Description, xóa dữ liệu thiếu InvoiceNo và chuyển InvoiceNo thành chuỗi

In [ ]:
print("Phạm Lê Anh Khoa - 6451071035")

df['Description'] = df['Description'].str.strip()
df.dropna(axis=0, subset=['InvoiceNo'], inplace=True)
df['InvoiceNo'] = df['InvoiceNo'].astype('str')

df.head(10)

Câu 3. Xóa các hóa đơn tín dụng có ký tự 'C' trong InvoiceNo

In [ ]:
print("Phạm Lê Anh Khoa - 6451071035")
df[df['InvoiceNo'].str.contains('C', na=False)].head(10)

In [ ]:
print("Phạm Lê Anh Khoa - 6451071035")
df = df[~df['InvoiceNo'].str.contains('C', na=False)].copy()
df.head(10)


Câu 4. Thống kê số dòng dữ liệu theo từng quốc gia (Country)

In [ ]:
print("Phạm Lê Anh Khoa - 6451071035")
df['Country'].value_counts().plot(kind='barh', figsize=(15, 10))

Câu 5. Lấy hóa đơn từ United Kingdom và gom nhóm Quantity theo InvoiceNo và Description

In [ ]:

print("Phạm Lê Anh Khoa - 6451071035")

basket = (
    df[df['Country'] == 'United Kingdom']
    .groupby(['InvoiceNo', 'Description'])['Quantity']
)

basket.head(10)


Câu 6. Chuyển đổi dữ liệu về dạng Hot Encoding

In [ ]:
print("Phạm Lê Anh Khoa - 6451071035")
basket = basket.sum().unstack().reset_index().fillna(0).set_index('InvoiceNo')
basket.head(10)

Câu 7. Chuyển đổi dữ liệu Hot Encoding thành One-hot Encoding

In [ ]:
print("Phạm Lê Anh Khoa - 6451071035")

def encode_data(datapoint):
    if datapoint <= 0:
        return 0
    if datapoint >= 1:
        return 1

basket = basket.map(encode_data)

basket.head(10)


Câu 8. Xóa thuộc tính POSTAGE

In [ ]:
print("Phạm Lê Anh Khoa - 6451071035")
basket.drop('POSTAGE', inplace=True, axis=1,errors='ignore')
basket.head(10)

Câu 9. Tìm các tập phổ biến bằng thuật toán Apriori với min_sup = 3%

In [ ]:

print("Phạm Lê Anh Khoa - 6451071035")

itemsets = apriori(
    basket.astype(bool),
    min_support=0.03,
    use_colnames=True
)

itemsets.head(10)


Câu 10. Tạo các luật kết hợp với min_conf = 50% và in ra kết quả

In [ ]:
print("Phạm Lê Anh Khoa - 6451071035")

rules = association_rules(
    itemsets,
    metric="confidence",
    min_threshold=0.5
)

rules.info()


Chuyển đổi vế trái và vế phải từ kiểu object (frozenset) về kiểu chuỗi (unicode)

In [ ]:
print("Phạm Lê Anh Khoa - 6451071035")
rules["antecedents"] = rules["antecedents"].apply(
    lambda x: list(x)[0]
).astype(str)

rules["consequents"] = rules["consequents"].apply(
    lambda x: list(x)[0]
).astype(str)

rules.head(10)


Viết lệnh in ra các luật đã tìm được

In [ ]:
print("Phạm Lê Anh Khoa - 6451071035")

for i in range(len(rules)):
    print(
        rules.loc[i, 'antecedents'], '==>',
        rules.loc[i, 'consequents'],
        '[',
        rules.loc[i, 'support'], ',',
        rules.loc[i, 'confidence'],
        ']'
    )

Câu 11. Biểu diễn độ hỗ trợ (Support) và độ tin cậy (Confidence) bằng Scatter Plot

In [ ]:

print("Phạm Lê Anh Khoa - 6451071035")

support = rules['support'].values
confidence = rules['confidence'].values

plt.figure(figsize=(6, 6))
plt.title('Association Rules')
plt.xlabel('support')
plt.ylabel('confidence')

sns.regplot(
    x=support,
    y=confidence,
    fit_reg=False
)

plt.show()


Câu 12. Tìm tập phổ biến và luật kết hợp bằng FP-Growth với min_sup = 3%, min_conf = 50% và so sánh với Apriori

1. Tìm tập phổ biến bằng FP-Growth

In [ ]:

from mlxtend.frequent_patterns import fpgrowth

print("Phạm Lê Anh Khoa - 6451071035")

itemsets_fp = fpgrowth(
    basket.astype(bool),
    min_support=0.03,
    use_colnames=True
)

itemsets_fp.head(10)


2. Tạo luật kết hợp bằng FP-Growth

In [ ]:
print("Phạm Lê Anh Khoa - 6451071035")
rules_fp = association_rules(
    itemsets_fp,
    metric="confidence",
    min_threshold=0.5
)

rules_fp.info()


3. Chuyển đổi vế trái và vế phải thành chuỗi

In [ ]:
print("Phạm Lê Anh Khoa - 6451071035")
rules_fp["antecedents"] = rules_fp["antecedents"].apply(
    lambda x: ', '.join(sorted(x))
)

rules_fp["consequents"] = rules_fp["consequents"].apply(
    lambda x: ', '.join(sorted(x))
)

rules_fp.head(10)


4. In ra các luật kết hợp đã tìm được



In [ ]:
print("Phạm Lê Anh Khoa - 6451071035")

for i in range(len(rules_fp)):
    print(
        rules_fp.iloc[i]['antecedents'], '==>',
        rules_fp.iloc[i]['consequents'],
        '[',
        rules_fp.iloc[i]['support'], ',',
        rules_fp.iloc[i]['confidence'],
        ']'
    )

5. Vẽ biểu đồ Support và Confidence

In [ ]:
print("Phạm Lê Anh Khoa - 6451071035")
support_fp = rules_fp['support'].values
confidence_fp = rules_fp['confidence'].values

plt.figure(figsize=(6, 6))
plt.title('Association Rules - FP-Growth')
plt.xlabel('support')
plt.ylabel('confidence')

sns.regplot(
    x=support_fp,
    y=confidence_fp,
    fit_reg=False
)

plt.show()
